In [ ]:
import uuid, time, json
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from utils.logging_config import get_logger
from pathlib import Path
from typing import List, Dict


logger = get_logger(__name__)



In [ ]:
from config.settings import Settings
cfg = Settings()

In [13]:
def get_embeddings():
    print("Loading embedding model...")
    try:
        return SentenceTransformer(
            cfg.embeddings.model_name
        )

    except Exception as e:
        print(f"failed to load model due to {e}")
        return "Fuck you"

In [14]:
model = get_embeddings()

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 780.58it/s]
BertModel LOAD REPORT from: models\embedding_ml\all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
texts = 'Hey what'

In [18]:
embeddings = model.encode(
            texts,
            normalize_embeddings=True,
            )
type(embeddings)

numpy.ndarray

In [ ]:
get_embeddings()

In [ ]:
client = QdrantClient(url="http://localhost:6333", prefer_grpc=True, timeout=120)
Collection = '9jalawya'
EMBEDDING_MODEL_NAME = "models\\embedding_ml\\all-MiniLM-L6-v2"
SPARSE_MODEL_NAME = "Qdrant/bm25"
CHUNKED_DATA_DIR = Path("data\chunked")
BATCH_SIZE = 256

In [ ]:
# Collection Creation

def setup_qdrant_collection(
    collection_name: str = Collection, 
    client: QdrantClient = client, 
    reset: bool = False
) -> bool:
    """
    Set up the Qdrant collection if it doesn't exist.
    Returns True if created, False if it already existed.
    """
    logger.info(f"Setting up Qdrant collection: {collection_name}")
    created = False

    if reset and client.collection_exists(collection_name=collection_name):
        try:
            client.delete_collection(collection_name=collection_name)
            logger.info(f"Collection {collection_name} deleted successfully(reset=True).")
        except Exception:
            pass

    if not client.collection_exists(collection_name=collection_name):
        logger.info(f"Collection {collection_name} does not exist. Creating collection...")
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "dense": models.VectorParams(
                    size=384,
                    distance=models.Distance.COSINE,
                )
            },
            sparse_vectors_config={
                "sparse": models.SparseVectorParams(
                    modifier=models.Modifier.IDF  # for BM25-style
                )
            },
        )
        logger.info(f"Collection {collection_name} created successfully.")
        created = True

    # Ensure payload index exists even if the collection was already created
    client.create_payload_index(
        collection_name=collection_name,
        field_name="doc_key",
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

    return created



In [ ]:
def load_embedding_model(
        model_name: str = EMBEDDING_MODEL_NAME
        ) -> SentenceTransformer:
    """
    Load the embedding model.
    Args:
        model_name: HuggingFace model identifier string
    
    Returns:
        Loaded SentenceTransformer model object
    """
    start = time.time()
    logger.info(f"Loading embedding model: {model_name}")
    try:
            model = SentenceTransformer(model_name)
            elapsed = round(time.time() - start, 2)
            logger.info(f"  Model loaded successfully in {elapsed}s")
            logger.info(f"  Embedding dimensions: {model.get_sentence_embedding_dimension()}")
           
            return model
    
    except Exception as e:
        logger.error(f"Failed to load embedding model '{model_name}': {e}")
        logger.error("Ensure sentence-transformers is installed: pip install sentence-transformers")
        raise



In [ ]:
def load_sparse_model(model_name: str = SPARSE_MODEL_NAME) -> SparseTextEmbedding:
    """
    Load the sparse embedding model (BM25).
    
    Returns:
        Loaded SparseTextEmbedding model object
    """
    start = time.time()
    logger.info("Loading sparse embedding model: Qdrant/bm25")
    try:
        sparse_model = SparseTextEmbedding(model_name)
        elapsed = round(time.time() - start, 2)
        logger.info(f"  Sparse model loaded successfully in {elapsed}s")
        return sparse_model
    
    except Exception as e:
        logger.error(f"Failed to load sparse embedding model: {e}")
        raise

# CHUNK LOADER
def load_all_chunks(chunked_dir: Path = CHUNKED_DATA_DIR) -> List[Dict]:
    """
    Loads all chunked JSON files and returns a flat list of all chunks.

    We flatten all chunks from all documents into one list because ChromaDB
    works with a single flat collection — it doesn't have a concept of
    sub-collections per document. Document identity is preserved through
    the doc_key metadata field on each chunk.

    Args:
        chunked_dir: Directory containing chunked JSON files

    Returns:
        Flat list of all chunk dictionaries from all documents
    """

    if not chunked_dir.exists():
        logger.error(f"Chunked data directory not found: {chunked_dir}")
        logger.error("Please run chunker.py first.")
        return []

    chunked_files = list(chunked_dir.glob("*_chunked.json"))

    if not chunked_files:
        logger.error(f"No chunked JSON files found in {chunked_dir}")
        return []

    logger.info(f"Loading chunks from {len(chunked_files)} document file(s)...")

    all_chunks = []

    for chunked_path in sorted(chunked_files):
        try:
            with open(chunked_path, "r", encoding="utf-8") as f:
                doc_data = json.load(f)

            chunks = doc_data.get("chunks", [])
            all_chunks.extend(chunks)
            # .extend() adds all items from chunks into all_chunks.
            # This is more efficient than calling .append() in a loop
            # because it adds all items in one operation.

            logger.info(
                f"  {doc_data.get('doc_key', 'unknown'):<35} "
                f"{len(chunks):>4} chunks loaded"
            )

        except Exception as e:
            logger.error(f"Failed to load {chunked_path.name}: {e}")
            continue

    logger.info(f"  Total chunks loaded: {len(all_chunks):,}")
    return all_chunks

# DEDUPLICATION CHECK
def filter_already_ingested(
    chunks: List[Dict],
    client: QdrantClient,
    collection_name: str
) -> List[Dict]:
    """
    Filters out chunks that are already present in the Qdrant collection.

    This enables INCREMENTAL ingestion — if you add a new document and
    re-run ingestor.py, only the new chunks are ingested. Existing chunks
    are skipped rather than duplicated.

    HOW IT WORKS:
    Each chunk has a deterministic chunk_id generated by chunker.py based
    on its content. We check which of those point IDs already exist in Qdrant
    and skip them.

    Args:
        chunks: All chunks loaded from disk
        client: The initialized QdrantClient instance
        collection_name: The name of the Qdrant collection to check against

    Returns:
        List of chunks that are NOT yet in the collection (need ingesting)
    """
    if not chunks:
        return []

    # Check if the collection exists and whether it is empty.
    # exact=False provides a fast approximate count, which is sufficient for checking > 0.
    if not client.collection_exists(collection_name) or client.count(collection_name, exact=False).count == 0:
        # Collection doesn't exist or is empty — all chunks need to be ingested.
        # Skip the ID lookup entirely.
        logger.info("  Collection is empty — all chunks will be ingested")
        return chunks

    logger.info("  Checking for already-ingested chunks...")

    # Extract all chunk IDs from our loaded chunks.
    # Note: Qdrant point IDs must be valid UUID strings or unsigned 64-bit integers.
    all_ids = [chunk["chunk_id"] for chunk in chunks]

    # Query Qdrant to find which of these IDs already exist.
    # We use retrieve() with with_payload=False and with_vectors=False
    # so only the matching point IDs are returned, minimizing network transfer.
    try:
        existing_records = client.retrieve(
            collection_name=collection_name,
            ids=all_ids,
            with_payload=False,
            with_vectors=False
        )

        # Convert the returned Record IDs to a set for O(1) lookup performance.
        # Ensure string/int types match your chunk["chunk_id"] format.
        existing_ids = {str(record.id) for record in existing_records}

    except Exception as e:
        logger.warning(f"  Could not check existing IDs: {e}. Will ingest all chunks.")
        return chunks

    new_chunks = [c for c in chunks if str(c["chunk_id"]) not in existing_ids]
    # List comprehension: keep only chunks whose ID is NOT in existing_ids.

    skipped = len(chunks) - len(new_chunks)
    if skipped > 0:
        logger.info(f"  Skipping {skipped:,} already-ingested chunks")
    logger.info(f"  Chunks to ingest: {len(new_chunks):,}")

    return new_chunks

In [ ]:
setup_qdrant_collection(collection_name=Collection, client=client, reset=False)

In [ ]:
embedding = load_embedding_model()
sparse = load_sparse_model()
chunks = load_all_chunks()

In [ ]:
def ingest(chunks, batch_size=BATCH_SIZE, dense_model=None, sparse_model=None, client=client):
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]
        dense = dense_model.encode(texts, normalize_embeddings=True)
        sparse = list(sparse_model.embed(texts))

        points = [
            models.PointStruct(
                id=str(uuid.uuid5(uuid.NAMESPACE_URL, c["chunk_id"])),
                vector={
                    "dense": d.tolist(),
                    "sparse": models.SparseVector(
                        indices=s.indices.tolist(), values=s.values.tolist()
                    ),
                },
                payload={
                    "doc_key": c["doc_key"],
                    "section": c["section_number_int"],
                    "section_title": c["section_title"],
                    "text": c["text"],
                    "corpus_version": "2026-11",
                },
            )
            for c, d, s in zip(batch, dense, sparse)
        ]
        client.upsert(Collection, points=points)

In [ ]:
sparse

In [ ]:
ingest(chunks, batch_size=BATCH_SIZE, dense_model=embedding, sparse_model=sparse, client=client)

In [ ]:
def retrieve(query: str, doc_keys: list[str] | None = None, k: int = 20):
    flt = None
    if doc_keys:
        flt = models.Filter(must=[
            models.FieldCondition(key="doc_key", match=models.MatchAny(any=doc_keys))
        ])

    dense_q = embedding.encode(query, normalize_embeddings=True).tolist()
    sp = next(sparse.query_embed(query))
    sparse_q = models.SparseVector(indices=sp.indices.tolist(), values=sp.values.tolist())

    res = client.query_points(
        collection_name=Collection,
        prefetch=[
            models.Prefetch(query=dense_q, using="dense", limit=k, filter=flt),
            models.Prefetch(query=sparse_q, using="sparse", limit=k, filter=flt),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=k,
        with_payload=True,
    )
    return res.points   # each has .score and .payload

In [ ]:
retrieve(query = "What is the law on child custody in Nigeria?", doc_keys=None, k=5)